<div style="background:#E9FFF6; color:#440404; padding:8px; border-radius: 4px; text-align: center; font-weight: 500;">IFN619 - Data Analytics for Strategic Decision Makers</div>

# IFN619 :: B3-Semi/unstructured Analytics Tutorial Exercises (Part A)

For this tutorial, you will use the lecture notebooks as a guide, and:

1. Use the Guardian API to undertake your own search and obtain a json file of documents
2. Create a TF/IDF document-term matrix for your documents

In [1]:
# Import the necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
import json
import random

## 1. Accessing the data via The Guardian API

Make a copy of the lecture notebook file (Accessing the Guardian API), and modify it to perform your own search of the Guardian API. **NOTE:** you will need to obtain your own developer API key first and put it in a file in the appropriate folder.

A suggested topic is "Brisbane 2032 Olympics", or come up with another that is of interest to you and will return a fair amount of data.

Save your search results in a json file, then read in that data below...

In [2]:
# Load the data - articles from The Guardian
file_path = "data/"
file_name = "brisbane_olympics_articles.json"

with open(f"{file_path}{file_name}",'r', encoding='utf-8') as fp:
    articles = json.load(fp)

print(f"Loaded {len(articles)} articles from {file_name}")

Loaded 126 articles from brisbane_olympics_articles.json


#### Discussion
Let's have a quick look what articles we have collected. 

In [3]:
# An overview of article titles
for title in articles.keys():
    print(title)

Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]
‘A big call for the IOC’: is the fight over Olympic rowing on Australia’s predator-inhabited Fitzroy River all a croc?  [2026-02-14T19:00:01Z]
Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z]
North Melbourne beat Brisbane: AFLW 2025 grand final – as it happened [2025-11-29T11:18:30Z]
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z]
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z]
‘We’re seen as the underdogs’: the Australian skiers out on their own chasing an Olympic dream | Kieran Pender [2026-01-04T14:00:11Z]
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in 

Did all the retrieved articles relevant to the topic? If not, what are the strategies we might use to filter articles that are relevant to our topic? 
- ???
- ???
- ???

In [4]:
# Implement some of your strategies - add more cells if needed

# e.g., refine search terms and apply appropriate filters when building a search URL
# DIY 

# e.g., remove articles contains 'as it happened' - why?
## get a list of article titles 
titles = list(articles.keys())
## create an empty list to store filtered titles
filtered_titles = []

for title in titles:
    if "as it happens" not in title:
        filtered_titles.append(title)
        
# e.g., include titles that contain 'Brisbane' and 'Olympic' - why?
filtered_titles_2 = [title for title in filtered_titles if 'Brisbane' in title and 'Olympic' in title]

filtered_titles_2

['Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]',
 'Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]',
 'Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z]',
 'First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z]',
 'AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z]',
 '‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z]',
 'Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z]',
 '‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z]',
 'LNP denies de

Since we have filtered out some articles, we need to update the JSON file to remove the articles that are no longer relevant. If we don't do this, the code below might try to access articles that don't exist anymore, which could cause errors.

In [5]:
# Filter the JSON data to only include these titles
# advanced
# articles_filtered = {title: content for title, content in articles.items() if title in filtered_titles_2}

# an easier way
## Initialise an empty dictionary
articles_filtered = {}
## Loop through the articles and add the ones that are in the DataFrame's index
for title, content in articles.items():
    if title in filtered_titles_2:
        articles_filtered[title] = content

articles_filtered

{'Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]': 'If you typed the words “believe, belong and become” into a Google video search on Thursday morning, the first return may have been a sermon by TJ Mauldin, the lead pastor of the First Baptist church of Tifton, Georgia. Directly below the bearded and blue-jeaned pastor’s video under that alliterative banner, you may have clicked through to a sermon by West Florida Baptist church’s Mike Brown, who had those three b-words emblazoned on a snug-fitting black T-shirt. “These aren’t just words on a shirt,” the tanned, fit and immaculately groomed senior pastor proclaims. “This is the message and the heartbeat and the passion of God’s word – and it oughta be the message and the heartbeat and the passion of his church!”   Related: Can crocodiles and canoeists coexist at Australia’s 2032 Olympic Games?    From Lewis Center for Church Leadership Rev Dexter Udell N

In [6]:
len(articles_filtered)

13

#### Create a top10 terms dataframe

Using the index from the documents, create a dataframe that can hold the top10 terms for each document.

In [7]:
# Create a dataframe to hold top terms for each analysis type
# We are going to use the article titles as index
terms_df = pd.DataFrame(index=articles_filtered.keys(),columns=['count','tfidf'])
terms_df

,count,tfidf
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]",NaN,NaN
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]",NaN,NaN
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],NaN,NaN
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],NaN,NaN
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],NaN,NaN
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],NaN,NaN
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],NaN,NaN
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],NaN,NaN
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],NaN,NaN
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],NaN,NaN


## 2. Term Count

In [8]:
# Set parameters appropriate to your data
count_vectorizer = CountVectorizer(max_df=0.80,min_df=2,max_features=10000,stop_words="english")
count_dt_matrix = count_vectorizer.fit_transform(articles_filtered.values())

In [9]:
# Get the terms identified during the vectorization process
feature_names = count_vectorizer.get_feature_names_out()
feature_names

array(['000', '10', '100', '11', '12', '15', '1980s', '1990s', '1bn',
       '20', '2001', '2012', '2016', '2017', '2018', '2019', '2020',
       '2021', '2024', '2025', '2026', '2027', '23', '25', '27', '30',
       '40', '4bn', '50', '5bn', '63', '785bn', '8bn', 'abc', 'able',
       'aboriginal', 'absolutely', 'accepted', 'access', 'according',
       'acknowledges', 'act', 'acting', 'action', 'active', 'actually',
       'additional', 'address', 'advance', 'advertising', 'affairs',
       'affected', 'afl', 'afternoon', 'age', 'agencies', 'agenda', 'ago',
       'agree', 'agreement', 'ahead', 'albanese', 'alliance', 'allow',
       'alongside', 'alternative', 'amid', 'amp', 'andrew', 'angeles',
       'announce', 'announced', 'announcement', 'anthony', 'anti', 'aoc',
       'apologised', 'appear', 'appointed', 'appropriate', 'approval',
       'approve', 'approved', 'april', 'aquatic', 'arbib', 'area',
       'areas', 'aren', 'arena', 'argue', 'argued', 'argues',
       'arrangemen

In [10]:
# Create a new dataframe with the matrix - use titles for the index and terms for the columns
count_df = pd.DataFrame(count_dt_matrix.toarray(), index = terms_df.index, columns=feature_names)
count_df

,000,10,100,11,12,15,1980s,1990s,1bn,20,...,work,working,works,world,wouldn,writing,written,year,years,yes
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]",2,0,0,0,0,0,0,0,0,0,...,0,0,0,3,0,0,0,0,0,0
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]",0,0,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,2,0
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],0,0,1,0,0,1,0,0,0,0,...,0,0,1,1,0,0,1,0,0,0
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],2,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,3,0,0
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,2,0
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],1,0,0,0,0,2,1,1,0,0,...,0,1,0,5,1,0,0,1,3,0
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],2,0,0,0,0,0,0,0,1,0,...,1,0,1,0,0,0,0,2,0,0
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],2,1,0,0,0,1,0,0,0,0,...,0,0,1,2,0,1,1,1,0,1
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],6,21,1,18,11,2,0,0,0,4,...,17,7,1,9,1,1,0,16,14,1
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],5,0,0,0,2,0,0,0,2,1,...,1,0,0,2,0,0,0,2,1,0


In [11]:
#For each doc, get the 10 columns with the largest counts
for idx in terms_df.index:
    counts = dict(count_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'count'] = list(counts.keys()) # Just the list of terms

terms_df

,count,tfidf
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]","[words, says, believe, search, passion, video,...",NaN
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]","[says, story, people, history, men, today, com...",NaN
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],"[board, women, committee, planning, indigenous...",NaN
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],"[park, aboriginal, stadium, heritage, victoria...",NaN
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],"[arbib, aoc, sports, executive, athletes, comm...",NaN
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],"[crocodile, rowing, crocodiles, river, world, ...",NaN
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],"[new, park, site, victoria, bleijie, stadium, ...",NaN
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],"[park, victoria, stadium, people, plan, save, ...",NaN
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],"[police, people, australia, day, sydney, nsw, ...",NaN
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],"[new, stadium, centre, build, crisafulli, priv...",NaN


## 3. Term Frequency / Inverse Document Frequency (TF/IDF)


In [12]:
# Set parameters appropriate to your data
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.80, min_df=2, max_features=10000, stop_words="english"
)

In [13]:
# Get the document vectors
tfidf_dt_matrix = tfidf_vectorizer.fit_transform(articles_filtered.values())

# Display the vector for the first document
tfidf_dt_matrix.toarray()[0]

array([0.07189895, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.0585578 ,
       0.        , 0.        , 0.        , 0.04678316, 0.05192666,
       0.        , 0.        , 0.        , 0.        , 0.0585578 ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.05192666, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.04258061,
       0.        , 0.        , 0.05192666, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.17567339,
       0.        , 0.        , 0.        , 0.0585578 , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.0585578 , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.04258061, 0.        ,
       0.        , 0.03080597, 0.        , 0.        , 0.     

#### Update the terms matrix

In [14]:
# list of feature names
feature_names = tfidf_vectorizer.get_feature_names_out()

# create a df to combine matrix with feature names
tfidf_df = pd.DataFrame(tfidf_dt_matrix.toarray(), index=articles_filtered.keys(), columns=feature_names)
tfidf_df

,000,10,100,11,12,15,1980s,1990s,1bn,20,...,work,working,works,world,wouldn,writing,written,year,years,yes
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]",0.071899,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.099704,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.047781,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.033944,0.000000,0.000000,0.000000,0.000000,0.067887,0.000000
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],0.000000,0.000000,0.045773,0.000000,0.000000,0.050291,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.050291,0.035727,0.000000,0.000000,0.062949,0.000000,0.000000,0.000000
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],0.068617,0.049556,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.049556,0.000000,0.000000,0.088199,0.000000,0.000000
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.058108,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.030570,0.065959,0.000000
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],0.023403,0.000000,0.000000,0.000000,0.000000,0.060912,0.030456,0.038122,0.000000,0.000000,...,0.000000,0.038122,0.000000,0.108180,0.033805,0.000000,0.000000,0.020055,0.064908,0.000000
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],0.066247,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.047845,0.000000,...,0.043106,0.000000,0.043106,0.000000,0.000000,0.000000,0.000000,0.056769,0.000000,0.000000
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],0.061066,0.044103,0.000000,0.000000,0.000000,0.039734,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.039734,0.056454,0.000000,0.049735,0.049735,0.026164,0.000000,0.049735
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],0.022784,0.115186,0.004498,0.111339,0.060336,0.009883,0.000000,0.000000,0.000000,0.021940,...,0.084010,0.043299,0.004942,0.031595,0.005485,0.006186,0.000000,0.052065,0.049148,0.006186
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],0.140903,0.000000,0.000000,0.000000,0.081410,0.000000,0.000000,0.000000,0.081410,0.040705,...,0.036673,0.000000,0.000000,0.052105,0.000000,0.000000,0.000000,0.048297,0.026052,0.000000


In [15]:
for idx in terms_df.index:
    tfidf = dict(tfidf_df.loc[idx].sort_values(ascending=False).head(10))
    terms_df.at[idx,'tfidf'] = list(tfidf.keys()) 

terms_df

,count,tfidf
"Believe, belong, become, boring, bizarre: Brisbane Olympics motto panned as ‘lazy and weirdly evangelical’ [2025-12-04T06:05:28Z]","[words, says, believe, search, passion, video,...","[words, believe, says, advertising, passion, s..."
"Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]","[says, story, people, history, men, today, com...","[says, story, men, history, jones, england, to..."
Queensland to repeal diversity quotas for Brisbane Olympics board [2025-06-19T14:43:14Z],"[board, women, committee, planning, indigenous...","[board, women, directors, committee, planning,..."
First Nations group launches federal appeal to block construction of main Brisbane 2032 Olympics stadium [2025-08-05T03:39:12Z],"[park, aboriginal, stadium, heritage, victoria...","[park, aboriginal, heritage, federal, victoria..."
AOC appoints Mark Arbib as CEO for run-in to 2032 Brisbane Olympic Games [2025-04-01T23:52:32Z],"[arbib, aoc, sports, executive, athletes, comm...","[arbib, aoc, sports, executive, experience, at..."
‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z],"[crocodile, rowing, crocodiles, river, world, ...","[crocodile, rowing, crocodiles, river, behavio..."
Federal government throws support behind LNP’s controversial new Olympic Games venue at Brisbane’s Victoria Park [2025-07-03T07:13:33Z],"[new, park, site, victoria, bleijie, stadium, ...","[bleijie, new, victoria, park, site, king, dea..."
‘Uneasy alliance’: why Brisbane’s Olympics has a former LNP premier and Greens radical fighting on the same side [2025-10-11T23:00:36Z],"[park, victoria, stadium, people, plan, save, ...","[park, victoria, stadium, save, people, newman..."
LNP denies decision has been made on Brisbane Olympic venues – as it happened [2025-01-23T06:55:26Z],"[police, people, australia, day, sydney, nsw, ...","[police, people, australia, sydney, nsw, man, ..."
Brisbane Olympics 2032: David Crisafulli breaks election promise and announces controversial new stadium [2025-03-25T02:26:54Z],"[new, stadium, centre, build, crisafulli, priv...","[new, private, sector, build, stadium, arena, ..."


## 4. Compare approaches

In [16]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc.name}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print()

[11] Brisbane Olympics infrastructure body urges LNP to fast-track controversial venues with special laws [2025-03-26T14:00:02Z]
	>> Counts:	 ['review', 'venues', 'time', 'infrastructure', 'procurement', 'planning', 'park', 'victoria', 'special', 'processes']
	>> TFIDF:	 ['procurement', 'review', 'required', 'special', 'processes', 'projects', 'case', 'infrastructure', 'victoria', 'venues']

[1] Behind the scenes of Brisbane’s 2032 Olympics, a 19th-century story of politics and gay love [2026-01-02T14:00:13Z]
	>> Counts:	 ['says', 'story', 'people', 'history', 'men', 'today', 'community', 'jones', 'know', 'england']
	>> TFIDF:	 ['says', 'story', 'men', 'history', 'jones', 'england', 'today', 'people', 'unlawful', 'reads']

[5] ‘Lovely gentle dinosaurs’: Brisbane 2032 Olympic rowing may be held in saltwater crocodile habitat [2025-03-25T08:31:39Z]
	>> Counts:	 ['crocodile', 'rowing', 'crocodiles', 'river', 'world', 'just', 'international', 'city', 'australia', 'behaviour']
	>> TFIDF:	 [

## Refine your analysis

Once you have worked through the process. Try tweaking the parameters in the Count and TF/IDF vectorizers to try and obtain better results for your data.

#### Advanced

You may obtain better results by doing the following:

- Creating smaller documents (e.g. article paragraphs)
- Pre-processing the text by Stemming or Lemmatizing, and by removing additional stop words.
- ???

## Reflections

Based on your exploration, reflect on the following questions:

- Considering The Guardian focuses on stories with national or global significance, what kinds of articles can you find using the Guardian API? 
- Based on the articles you retrieved, what kind of narratives can you build, and which ones are impossible to support with this dataset?
- To enhance the narrative, what visualisations would you consider?